<div align="center">
  <a href="https://pageindex.ai/developer">
    <img src="https://github.com/user-attachments/assets/bae02956-6c4e-4a0b-adea-257b0be4aaa1" width="100%" alt="PageIndex — Vectorless, Reasoning-based RAG">
  </a>
  <p>
    <a href="https://pageindex.ai/developer">🌐 Website</a> &nbsp; · &nbsp;
    <a href="https://developer.pageindex.ai/">☁️ Cloud</a> &nbsp; · &nbsp;
    <a href="https://docs.pageindex.ai/">📖 Docs</a> &nbsp; · &nbsp;
    <a href="https://github.com/VectifyAI/PageIndex">⭐ GitHub</a> &nbsp; · &nbsp;
    <a href="https://pageindex.ai/contact">✉️ Contact</a>
  </p>
</div>

---

# Accuracy and Query Cost: PageIndex vs Vector RAG

**Ask 63 questions about NVIDIA's annual reports for fiscal 2022–2026 with PageIndex and with vector search on six embedding models, each at four top-k settings, grade every answer, and compare accuracy and cost per 1,000 queries.**

<div align="center">
  <img src="https://raw.githubusercontent.com/VectifyAI/PageIndex/main/cookbook/pageindex-vs-vector-10k/accuracy-by-embedding.png" width="90%" alt="Accuracy against cost per 1,000 queries on 63 questions. PageIndex answers 100.0% for $110. At top-k 50, vector search answers between 63.5% (OpenAI 3-large) and 90.5% (Voyage 4); the best vector result is Voyage 4 at top-k 100, 92.1% for $134.">
</div>

Everything but the embedding model stays the same: the same 512-token chunks, the same prompt and the same generation model, **GPT-6.0 sol** (medium reasoning), for every system. PageIndex uses no embedding model; it builds its trees with **GPT-6.0 luna**. A judge model grades each answer against the question's key points.

1. **Set up**  Install the libraries and choose the models.

2. **Get the filings**  Download the five 10-Ks from NVIDIA's investor site.

3. **Get the questions**  Load the 63 questions and their key points.

4. **Build the PageIndex library**  Index all five filings into one library.

5. **Build the vector indexes**  Chunk the filings once and embed the chunks with each model.

6. **Answer with vector search**  For each model, retrieve the top-k chunks and answer, for each k.

7. **Answer with PageIndex**  Let the agent pick the filing and read the pages it needs.

8. **Grade the answers**  Check each answer against the key points.

9. **Compare**  Accuracy for every model and top-k, next to PageIndex.

**Before you start:** You need an OpenRouter API key. A full run takes about 30 minutes and costs about \$110 on OpenRouter. To try it for less, keep fewer models in `EMBEDDING_MODELS`, drop 100 from `TOP_KS`, or set `N_QUESTIONS` lower. Answers are saved as they arrive, so rerunning a cell picks up where it stopped. If OpenRouter refuses requests with error 402 while your balance is low, lower `WORKERS`.

## 01 · Set up

Install the PageIndex SDK and the libraries for vector search.

In [ ]:
%pip install -q -U pageindex tiktoken pypdfium2 numpy tenacity

### Choose the models

Set your OpenRouter API key. Every call goes through LiteLLM, so you can also call the providers directly: drop the `openrouter/` prefix and set their keys instead.

In [ ]:
import os

# Every call goes through OpenRouter, so one key covers every model.
os.environ["OPENROUTER_API_KEY"] = "Your OpenRouter API key here"

MODEL = "openrouter/openai/gpt-6-sol"                   # generation model, for every system
INDEX_MODEL = "openrouter/openai/gpt-6-luna"            # builds the PageIndex trees
JUDGE_MODEL = "openrouter/anthropic/claude-opus-5.5"    # grades the answers

# The only thing that changes between the vector runs. Remove entries for a cheaper run.
EMBEDDING_MODELS = {
    "Gemini Embedding 2": "openrouter/google/gemini-embedding-2",
    "OpenAI 3-small": "openrouter/openai/text-embedding-3-small",
    "OpenAI 3-large": "openrouter/openai/text-embedding-3-large",
    "Voyage 4 Lite": "openrouter/voyageai/voyage-4-lite",
    "Voyage 4": "openrouter/voyageai/voyage-4",
    "Voyage 4 Large": "openrouter/voyageai/voyage-4-large",
}

REASONING_EFFORT = "medium"
TOP_KS = (10, 30, 50, 100)
N_QUESTIONS = 63    # all of them; fewer for a quick, cheaper run
WORKERS = 8         # questions answered at the same time

# OpenRouter USD per 1M tokens, checked 2026-10-20 at https://openrouter.ai/models.
# OpenRouter bills uncached input on prompts of 1,024+ tokens at the cache-write price,
# so "input" below is that price; every prompt in this notebook is longer than that.
PRICES = {
    "openai/gpt-6-sol": {"input": 2.50, "cached": 0.20, "output": 10.00},
    "openai/gpt-6-luna": {"input": 0.125, "cached": 0.01, "output": 0.50},
}
EMBEDDING_PRICES = {"Gemini Embedding 2": 0.20, "OpenAI 3-small": 0.02, "OpenAI 3-large": 0.13,
                    "Voyage 4 Lite": 0.02, "Voyage 4": 0.06, "Voyage 4 Large": 0.12}

## 02 · Get the filings

Download NVIDIA's 10-Ks for fiscal 2022 to 2026 as PDFs from its investor-relations site, and check each one matches the file this notebook was tested on.

In [1]:
import hashlib
import pathlib

import pypdfium2 as pdfium
import requests

# fiscal year: (PDF id on investor.nvidia.com, SHA-256 of the file this notebook was tested on)
FILINGS = {
    2022: ("ca04d49f-caab-436f-9e6f-e8493c8e8515", "61ce01bbcd3886fa512dce61613a242a95c081d60b2f8bc0583b2f89ccad40eb"),
    2023: ("4e9abe7b-fdc7-4cd2-8487-dc3a99f30e98", "89981bbfcd91e20498c1060d7efb39022ac8f85e639f2841091695885aa9f8a8"),
    2024: ("1cbe8fe7-e08a-46e3-8dcc-b429fc06c1a4", "536f66d7f1c3413abbf643e0a02bd0aab65639116fe630225f3f93529244658b"),
    2025: ("177440d5-3b32-4185-8cc8-95500a9dc783", "b67bd67a64488a54886de001c788bc5059a965fc6ef5b4d8b71624951e13df8e"),
    2026: ("e361e58a-7483-44f5-bc62-a9080ae6ec72", "a056b59e4170c0123217a8e861a39af4a4a48d2df0d27b641017a618a280a396"),
}
DATA = pathlib.Path("nvidia-10k")
DATA.mkdir(exist_ok=True)

PAGES = {}  # filing name -> text of each page
for fy, (pdf_id, sha256) in FILINGS.items():
    path = DATA / f"NVIDIA_{fy}_10K.pdf"
    if not path.exists():
        response = requests.get(f"https://d18rn0p25nwr6d.cloudfront.net/CIK-0001045810/{pdf_id}.pdf", timeout=120)
        response.raise_for_status()
        path.write_bytes(response.content)
    assert hashlib.sha256(path.read_bytes()).hexdigest() == sha256, f"{path.name} is not the file this notebook was tested on"
    pdf = pdfium.PdfDocument(path)
    PAGES[path.stem] = [page.get_textpage().get_text_range() for page in pdf]
    print(f"{path.name}: {len(PAGES[path.stem])} pages")

NVIDIA_2022_10K.pdf: 143 pages
NVIDIA_2023_10K.pdf: 169 pages
NVIDIA_2024_10K.pdf: 96 pages
NVIDIA_2025_10K.pdf: 130 pages
NVIDIA_2026_10K.pdf: 93 pages


## 03 · Get the questions

Each question comes with a reference answer and the key points an answer must state. 15 questions name no filing, so the system has to find it; the other 48 name a fiscal year, and 16 of those compare two years.

In [2]:
import json

QUESTIONS_URL = "https://raw.githubusercontent.com/VectifyAI/PageIndex/main/cookbook/pageindex-vs-vector-10k/nvidia-10k-questions.json"

path = DATA / "questions.json"
if not path.exists():
    response = requests.get(QUESTIONS_URL, timeout=60)
    response.raise_for_status()
    path.write_text(response.text)
QUESTIONS = json.loads(path.read_text())[:N_QUESTIONS]
print(f"{len(QUESTIONS)} questions. The first one:")
print(json.dumps(QUESTIONS[0], indent=2))

63 questions. The first one:
{
  "id": "md_1",
  "type": "Compare two fiscal years",
  "filings": [
    "NVIDIA_2022_10K",
    "NVIDIA_2025_10K"
  ],
  "question": "Which independent subcontractors did NVIDIA's 10-K for fiscal year 2022 say handled the assembly, test and packaging work for most of NVIDIA's products, and which companies does NVIDIA's 10-K for fiscal year 2025 name for that work on its final products?",
  "answer": "FY2022 10-K: Amkor Technology, King Yuan Electronics, Omni Logistics, Siliconware Precision Industries and Wistron. FY2025 10-K: Hon Hai Precision Industry, Wistron and Fabrinet.",
  "key_points": [
    "The FY2022 10-K names Amkor Technology, King Yuan Electronics, Omni Logistics, Siliconware Precision Industries and Wistron for assembly, testing and packaging.",
    "The FY2025 10-K names Hon Hai Precision Industry, Wistron and Fabrinet for assembly, testing and packaging of final products."
  ]
}


A helper that answers many questions at once and saves each answer as it arrives.

In [ ]:
import random
import threading
from concurrent.futures import ThreadPoolExecutor

import logging

import litellm

litellm.suppress_debug_info = True
# LiteLLM's background logger complains when calls come from several threads at once;
# the calls themselves are unaffected, so keep its messages out of the output.
for name in ("LiteLLM", "asyncio"):
    logging.getLogger(name).setLevel(logging.CRITICAL)


def collect(file_name, answer_fn, items):
    """Run answer_fn over the items not yet in file_name, saving rows as they arrive.
    Rows with an "error" are retried on the next run; the last row per (id, system) wins."""
    path = DATA / file_name
    rows = [json.loads(line) for line in path.open()] if path.exists() else []
    done = {r["id"] for r in rows if "error" not in r}
    todo = [item for item in items if item["id"] not in done]
    lock = threading.Lock()

    def work(item):
        new = answer_fn(item)
        with lock, path.open("a") as f:
            f.writelines(json.dumps(r) + "\n" for r in new)

    with ThreadPoolExecutor(WORKERS) as pool:
        for n, _ in enumerate(pool.map(work, todo), 1):
            print(f"\r{file_name}: {n}/{len(todo)} new", end="", flush=True)
    print(f"\r{file_name}: {len(items)} questions done")
    rows = [json.loads(line) for line in path.open()]
    return list({(r["id"], r["system"]): r for r in rows}.values())

## 04 · Build the PageIndex library

Index all five filings into one local library. PageIndex reads each PDF's layout to build a tree of sections, then writes a short summary for every node.

In [3]:
from pageindex import PageIndexClient

client = PageIndexClient(index_model=INDEX_MODEL, chat_model=MODEL,
                         storage_path="pageindex-library", summary_max_words=70)


def index(name):
    try:
        return client.get_document_id(f"{name}.pdf")  # already in the library
    except Exception:
        return client.submit_document(str(DATA / f"{name}.pdf"))["doc_id"]


with ThreadPoolExecutor(len(PAGES)) as pool:
    DOC_IDS = dict(zip(PAGES, pool.map(index, PAGES)))
print(f"{len(DOC_IDS)} filings in the library")

5 filings in the library


## 05 · Build the vector indexes

Split the filings into 512-token chunks with a 64-token overlap and embed the same chunks with each model, all five filings in one index per model.

In [4]:
import numpy as np
import tiktoken

enc = tiktoken.get_encoding("o200k_base")
CHUNK, OVERLAP = 512, 64


def chunks_of(name):
    """512-token windows with a 64-token overlap over the whole filing, keeping the pages each spans."""
    tokens, page_of = [], []
    for page, text in enumerate(PAGES[name], 1):
        ids = enc.encode(text)
        tokens += ids
        page_of += [page] * len(ids)
    for start in range(0, max(len(tokens) - OVERLAP, 1), CHUNK - OVERLAP):
        end = min(start + CHUNK, len(tokens))
        yield {"filing": name, "pages": (page_of[start], page_of[end - 1]), "text": enc.decode(tokens[start:end])}


def embed(model, texts):
    vectors, tokens = [], 0
    for i in range(0, len(texts), 100):
        response = litellm.embedding(model=model, input=texts[i:i + 100], num_retries=3)
        vectors += [d["embedding"] for d in sorted(response.data, key=lambda d: d["index"])]
        tokens += response.usage.prompt_tokens
    matrix = np.asarray(vectors, dtype=np.float32)
    return matrix / np.linalg.norm(matrix, axis=1, keepdims=True), tokens


def slug(name):
    return name.lower().replace(" ", "-")


CHUNKS = [chunk for name in PAGES for chunk in chunks_of(name)]
VECTORS = {}  # embedding model -> one unit vector per chunk, the same chunks for every model
for name, model in EMBEDDING_MODELS.items():
    cache = DATA / f"vectors_{slug(name)}.npy"
    if not cache.exists():
        np.save(cache, embed(model, [c["text"] for c in CHUNKS])[0])
    VECTORS[name] = np.load(cache)
print(f"{len(CHUNKS):,} chunks from {len(PAGES)} filings, embedded with {len(VECTORS)} models")

1,056 chunks from 5 filings, embedded with 6 models


## 06 · Answer with vector search

For each model and each question, find the chunks closest to the question and answer from the top 10, 30, 50 and 100. Every excerpt carries its filing name and pages.

In [5]:
PROMPT = """Answer the question using the excerpts below from a library of company \
filings. Each excerpt is labeled with its source filing and page numbers.

{context}

Question: {question}"""


def vector_answers(name):
    """For one embedding model: one query embedding and search, then one answer per k."""
    def answer(q):
        query, embedding_tokens = embed(EMBEDDING_MODELS[name], [q["question"]])
        order = np.argsort(-(VECTORS[name] @ query[0]))
        rows = []
        for k in TOP_KS:
            context = "\n\n".join(f"[Excerpt {n} | {c['filing']} | pages {c['pages'][0]}-{c['pages'][1]}]\n{c['text']}"
                                   for n, c in enumerate((CHUNKS[i] for i in order[:k]), 1))
            response = litellm.completion(model=MODEL, reasoning_effort=REASONING_EFFORT, num_retries=3, messages=[
                {"role": "user", "content": PROMPT.format(context=context, question=q["question"])}])
            usage = response.usage
            rows.append({"id": q["id"], "system": f"{name} top-{k}", "answer": response.choices[0].message.content,
                         "input_tokens": usage.prompt_tokens, "output_tokens": usage.completion_tokens,
                         "embedding_tokens": embedding_tokens})
        return rows
    return answer


VECTOR_ROWS = [row for name in EMBEDDING_MODELS
               for row in collect(f"answers_vector_{slug(name)}.jsonl", vector_answers(name), QUESTIONS)]

answers_vector_gemini-embedding-2.jsonl: 63 questions done
answers_vector_openai-3-small.jsonl: 63 questions done
answers_vector_openai-3-large.jsonl: 63 questions done
answers_vector_voyage-4-lite.jsonl: 63 questions done
answers_vector_voyage-4.jsonl: 63 questions done
answers_vector_voyage-4-large.jsonl: 63 questions done


## 07 · Answer with PageIndex

Ask the whole library. The agent picks the filing, reads its tree and fetches the pages it needs.

In [6]:
def pageindex_answer(q):
    """doc_id=None searches the whole library: the agent picks the filing itself."""
    try:
        response = client.chat(q["question"], doc_id=None, protocol="chat_completions",
                               reasoning_effort=REASONING_EFFORT, max_turns=20)
    except Exception as exc:
        return [{"id": q["id"], "system": "PageIndex", "error": f"{type(exc).__name__}: {exc}"[:300]}]
    usage = response["usage"]
    return [{"id": q["id"], "system": "PageIndex", "answer": response["choices"][0]["message"]["content"],
             "input_tokens": usage["prompt_tokens"], "output_tokens": usage["completion_tokens"],
             "cached_tokens": (usage.get("prompt_tokens_details") or {}).get("cached_tokens") or 0}]


PAGEINDEX_ROWS = collect("answers_pageindex.jsonl", pageindex_answer, QUESTIONS)

answers_pageindex.jsonl: 63 questions done


## 08 · Grade the answers

An answer is correct when it states every key point and contradicts nothing in the reference. A figure from the wrong fiscal year counts as a contradiction. The judge sees all 25 answers to a question at once, in random order, without knowing which system wrote which.

In [7]:
RUBRIC = """You are grading answers to a question about NVIDIA's annual reports on Form 10-K \
(fiscal years 2022 to 2026). You get the question, a reference answer, its key points, and \
several answers labeled A, B, C and so on. Judge each answer only against the reference and the \
key points. The question explicitly asks for every item the key points require.

Covering each key point
- Go through the key points in order and decide for each whether the answer states it.
- Numbers may differ only by rounding, and equivalent wording counts. Where the question says \
"approximately", accept figures within about 2% of the key point.
- A key point is NOT covered if the answer only hints at it, hedges between alternatives, or \
gives a materially different number.
- Every part of a key point outside parentheses must be stated. Parenthetical extras are \
required only if the question explicitly asks for them.
- A key point joining two facts with "and" needs both. Rounding that keeps the meaning counts.

Contradictions
- Decide whether the answer makes a claim that contradicts the reference: a wrong figure or \
reason presented as the answer.
- When the question names a fiscal year or a specific filing, an answer that gives another \
fiscal year's figure or statement, or the same year's figure as restated in a different filing, \
is a contradiction, even if that figure appears somewhere in NVIDIA's filings.
- Omissions, "not available" statements and extra correct detail are not contradictions.

General
- Do not reward length, confidence or formatting.
- An answer saying the information is not available covers nothing.
- Grade each answer on its own, with the same standard for all of them.

Reply with JSON only, one entry per answer:
{"grades": [{"label": "A", "covered": [true, false], "contradiction": false}]}
"covered" has one boolean per key point, in order."""

ANSWERS = {}
for row in VECTOR_ROWS + PAGEINDEX_ROWS:
    ANSWERS.setdefault(row["id"], {})[row["system"]] = row
SYSTEMS = ["PageIndex"] + [f"{name} top-{k}" for name in EMBEDDING_MODELS for k in TOP_KS]


def grade(q):
    """All answers to one question go to the judge together, shuffled and unlabeled."""
    systems = random.Random(q["id"]).sample(SYSTEMS, len(SYSTEMS))
    labels = [chr(ord("A") + n) for n in range(len(systems))]
    key_points = "\n".join(f"{n}. {p}" for n, p in enumerate(q["key_points"], 1))
    shown = "\n\n".join(f"Answer {label}:\n{ANSWERS[q['id']][s].get('answer') or '(no answer)'}"
                        for label, s in zip(labels, systems))
    prompt = (f"{RUBRIC}\n\nQuestion: {q['question']}\n\nReference answer: {q['answer']}"
              f"\n\nKey points:\n{key_points}\n\n{shown}")
    for attempt in range(3):  # retry the rare reply that is not valid JSON or skips an answer
        response = litellm.completion(model=JUDGE_MODEL, num_retries=3, messages=[{"role": "user", "content": prompt}])
        text = response.choices[0].message.content or ""
        try:
            grades = {g["label"]: g for g in json.loads(text[text.find("{"):text.rfind("}") + 1])["grades"]}
            assert all(len(grades[label]["covered"]) == len(q["key_points"]) for label in labels)
            break
        except (ValueError, KeyError, AssertionError):
            if attempt == 2:
                raise
    return [{"id": q["id"], "system": s, "contradiction": grades[label]["contradiction"],
             "correct": "answer" in ANSWERS[q["id"]][s] and all(grades[label]["covered"]) and not grades[label]["contradiction"]}
            for label, s in zip(labels, systems)]


GRADES = collect("grades.jsonl", grade, QUESTIONS)

grades.jsonl: 63 questions done


## 09 · Compare

Accuracy and cost per 1,000 queries for PageIndex and for each embedding model at each top-k, from the tokens each answer used.

In [8]:
def cost(row):
    """USD for one answer at the OpenRouter prices in step 01."""
    price = PRICES[MODEL.removeprefix("openrouter/")]
    # PageIndex's turns on one question reuse each other's prompt prefix. Vector search
    # answers each question with one call in a deployment, so it gets no cache discount.
    cached = row["cached_tokens"] if row["system"] == "PageIndex" else 0
    embedding_price = EMBEDDING_PRICES.get(row["system"].rsplit(" top-", 1)[0], 0)
    usd = ((row["input_tokens"] - cached) * price["input"] + cached * price["cached"]
           + row["output_tokens"] * price["output"] + row.get("embedding_tokens", 0) * embedding_price)
    return usd / 1e6


answered = {(r["id"], r["system"]): r for r in VECTOR_ROWS + PAGEINDEX_ROWS if "answer" in r}
correct = {s: sum(g["correct"] for g in GRADES if g["system"] == s) for s in SYSTEMS}
per_1k = {s: 1000 * sum(cost(r) for (_, name), r in answered.items() if name == s)
             / sum(name == s for _, name in answered) for s in SYSTEMS}
n = len(QUESTIONS)
print(f"PageIndex: {correct['PageIndex'] / n:.1%} of {n} questions, ${per_1k['PageIndex']:,.2f} per 1,000 queries")
for title, value in (("Accuracy", lambda s: f"{correct[s] / n:.1%}"),
                     ("Cost per 1,000 queries", lambda s: f"${per_1k[s]:,.2f}")):
    print(f"\n{title:25}" + "".join(f"{f'Top-k = {k}':>13}" for k in TOP_KS))
    for name in EMBEDDING_MODELS:
        print(f"{name:25}" + "".join(f"{value(f'{name} top-{k}'):>13}" for k in TOP_KS))

PageIndex: 100.0% of 63 questions, $110.48 per 1,000 queries

Accuracy                    Top-k = 10   Top-k = 30   Top-k = 50  Top-k = 100
Gemini Embedding 2               55.6%        76.2%        82.5%        85.7%
OpenAI 3-small                   52.4%        58.7%        65.1%        73.0%
OpenAI 3-large                   42.9%        61.9%        63.5%        77.8%
Voyage 4 Lite                    65.1%        79.4%        82.5%        85.7%
Voyage 4                         71.4%        85.7%        90.5%        92.1%
Voyage 4 Large                   76.2%        84.1%        88.9%        90.5%

Cost per 1,000 queries      Top-k = 10   Top-k = 30   Top-k = 50  Top-k = 100
Gemini Embedding 2              $15.77       $41.78       $68.26      $134.71
OpenAI 3-small                  $15.92       $42.69       $68.86      $135.12
OpenAI 3-large                  $16.53       $42.47       $68.71      $134.78
Voyage 4 Lite                   $15.37       $41.71       $67.91      $134.19
V

**How the cost is counted.** At the OpenRouter prices in step 01. Vector search is priced without the prompt cache, because a deployment answers each question with one call; it includes the query embedding. PageIndex keeps the cache hits between the agent's turns on one question, which any deployment gets too.

**About the embeddings.** Every model embeds questions and chunks as they are, without a query instruction.

**About the judge.** Claude Opus 5.5 grades every answer with the rubric above.

**About PageIndex's numbers.** They come from our earlier run of [pageindex-vs-vector-10k.ipynb](https://github.com/VectifyAI/PageIndex/blob/main/cookbook/pageindex-vs-vector-10k/pageindex-vs-vector-10k.ipynb); its answers are in [answers_pageindex.jsonl](https://github.com/VectifyAI/PageIndex/blob/main/cookbook/pageindex-vs-vector-10k/answers_pageindex.jsonl).

---

<div align="center">
  <br>
  <p>
    <a href="https://docs.pageindex.ai/"><b>Upload your documents</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://docs.pageindex.ai/sdk/agents"><b>Integrate with your agent</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://github.com/VectifyAI/PageIndex"><b>Star PageIndex on GitHub</b></a>
  </p>
  <br>
  <a href="https://pageindex.ai">
    <img src="https://github.com/user-attachments/assets/e1c677f2-b590-4ffc-859d-e8cc7d794bdf" width="144" height="40" alt="PageIndex">
  </a>
</div>
